In [ ]:
# ============================================================
# CELL 1 — Install Dependencies
# ============================================================
!pip install numpy==1.26.4
!pip install fasttext --upgrade

In [ ]:
import fasttext

In [ ]:
# ============================================================
# CELL 2 — Imports
# ============================================================
import re
import os
import numpy as np
import pandas as pd
import fasttext
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle


In [ ]:
# ============================================================
# CELL 3 — Load & Shuffle Dataset
# ============================================================
df = pd.read_csv("/content/lang_identification_1.csv")
df = shuffle(df, random_state=42).reset_index(drop=True)

print(df.head())
print("\nLabel distribution:")
print(df['label'].value_counts())


In [ ]:
# ============================================================
# CELL 4 — Prepare FastText Format & Train/Val Split
# ============================================================
df['fasttext_format'] = "__label__" + df['label'] + " " + df['text'].astype(str)

train_texts, val_texts = train_test_split(
    df['fasttext_format'],
    test_size=0.2,
    random_state=42,
    shuffle=True
)

with open("train.txt", "w", encoding="utf-8") as f:
    for line in train_texts:
        f.write(line + "\n")

with open("valid.txt", "w", encoding="utf-8") as f:
    for line in val_texts:
        f.write(line + "\n")

print(f"Train samples : {len(train_texts)}")
print(f"Val   samples : {len(val_texts)}")

In [ ]:
# ============================================================
# CELL 5 — Train FastText Model
# ============================================================
# FastText subword n-grams (minn/maxn) handle short tokens well,
# so no separate SVM fallback is needed.
model = fasttext.train_supervised(
    input="train.txt",
    lr=0.5,         # learning rate
    epoch=50,       # passes over dataset
    wordNgrams=2,   # bigrams for context
    dim=300,        # embedding dimension
    minn=2,         # min subword length
    maxn=5,         # max subword length  ← handles OOV & Roman Nepali
    bucket=200000,  # subword hash buckets
    loss='softmax', # standard multi-class
    thread=4,
)


In [ ]:
# ============================================================
# CELL 6 — Evaluate on Validation Set
# ============================================================
result = model.test("valid.txt")
print(f"Samples   : {result[0]}")
print(f"Precision : {result[1]:.4f}")
print(f"Recall    : {result[2]:.4f}")

In [ ]:
# ============================================================
# CELL 7 — Save Model
# ============================================================
model.save_model("fasttext_language_model.bin")
print("Model saved → fasttext_language_model.bin")


In [ ]:
# ============================================================
# CELL 8 — Load Model & Define Detection Function
# ============================================================
ft_model = fasttext.load_model("fasttext_language_model.bin")

CONFIDENCE_THRESHOLD = 0.5

def clean_text_for_inference(text: str) -> str:
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def predict_fasttext(text: str):
    labels, probs = ft_model.predict(text)
    label = list(labels)[0].replace("__label__", "")
    confidence = float(np.asarray(probs)[0])
    return label, confidence

def detect_language(text: str) -> dict:
    text = clean_text_for_inference(text)
    wc = len(text.split())

    if wc < 2:
        return {
            "text": text,
            "predicted_label": "UNKNOWN",
            "confidence": 0.0,
            "word_count": wc
        }

    label, confidence = predict_fasttext(text)

    if confidence < CONFIDENCE_THRESHOLD:
        label = "UNKNOWN"

    return {
        "text": text,
        "predicted_label": label,
        "confidence": round(confidence, 4),
        "word_count": wc
    }

# Quick smoke test
print(detect_language("yo sarkar ramrooooo cha"))
print(detect_language("सरकारले राम्रो काम गर्यो"))
print(detect_language("your government is strong"))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Prepare data for confusion matrix
true_labels = []
predicted_labels = []

with open("valid.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # Extract true label (e.g., __label__EN text -> EN)
        true_label = line.split(' ')[0].replace('__label__', '')
        text_to_predict = ' '.join(line.split(' ')[1:])

        # Get prediction from the FastText model
        pred_label, pred_confidence = predict_fasttext(text_to_predict)

        true_labels.append(true_label)
        predicted_labels.append(pred_label)

# Get all unique labels to ensure the confusion matrix has consistent order
all_labels = sorted(list(set(true_labels + predicted_labels)))

# Calculate the confusion matrix
cm = confusion_matrix(true_labels, predicted_labels, labels=all_labels)

# Plot the confusion matrix
fig, ax = plt.subplots(figsize=(10, 10))
display_cm = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=all_labels)
display_cm.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Confusion Matrix for Language Detection')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=45)
plt.tight_layout()
plt.savefig('confusion_matrix.png') # Save the figure before showing
plt.show()

In [ ]:
plt.savefig('confusion_matrix.png')
print("Confusion matrix plot saved as 'confusion_matrix.png'")

In [ ]:
# ============================================================
# CELL 9 — Run Detection on Comment List → DataFrame
# ============================================================
comments = [
    "yo sarkar ramro cha",
    "your government is strong",
    "ramro desh ho",
    "सरकारले राम्रो काम गर्यो",
]

rows = []
for comment in comments:
    result = detect_language(comment)
    rows.append({
        "raw_text":          comment,
        "detected_language": result["predicted_label"],
        "confidence":        result["confidence"],
        "word_count":        result["word_count"],
    })

df_results = pd.DataFrame(rows)
df_results


In [ ]:
# ============================================================
# CELL 10 — Save Model to Google Drive (models folder)
# ============================================================
from google.colab import drive
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Define the destination path inside your Drive 'models' folder
drive_models_path = "/content/drive/MyDrive/models"
os.makedirs(drive_models_path, exist_ok=True)

# Copy the saved model to Drive
model_filename = "fasttext_language_model.bin"
destination = os.path.join(drive_models_path, model_filename)

shutil.copy(model_filename, destination)
print(f"✅ Model saved to Google Drive: {destination}")